# Eksploracja danych · Laboratorium 6
## Analiza koszyka: reguły asocjacyjne

Sklep osiedlowy EconoShop Express, paragony z września i października 2026 r. (dane syntetyczne). Pytanie kierownika: **które produkty kupowane są razem i co z tym zrobić** (ułożenie półek, oferta „kupujący X biorą też Y”, promocje)?

**Najpierw** uzupełnij TODO 1–3 w `koszyk.py` i uruchom w terminalu `pytest -q` (cel: `7 passed`). Potem uruchamiaj komórki po kolei (Shift + Enter). Luki `___` w komórkach uzupełnij przed uruchomieniem.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import koszyk as k

df = k.wczytaj("dane/paragony.csv")          # jeden wiersz = jeden produkt na paragonie
K = k.macierz_koszykow(df)                    # paragon × produkt: True / False
info = df.groupby("id_paragonu").agg(data=("data", "first"), godzina=("godzina", "first")).reindex(K.index)
pd.set_option("display.precision", 3)

def para(A, B, K=K):
    """Wszystkie miary reguły A → B w jednym wierszu."""
    return {"A": A, "B": B, "paragony z A": k.liczba(K, A), "paragony z A i B": k.liczba(K, [A, B]),
            "wsparcie A i B": round(k.wsparcie(K, [A, B]), 4), "pewność": round(k.pewnosc(K, A, B), 3),
            "wsparcie B": round(k.wsparcie(K, B), 3), "lift": round(k.lift(K, A, B), 2)}

## Z1. Dane: ile paragonów, ile produktów, co jest popularne

In [ ]:
print("wierszy:", len(df), " paragonów:", K.shape[0], " produktów:", K.shape[1])
print("średnio produktów na paragonie:", round(K.sum(axis=1).mean(), 2))
wsp = K.mean().sort_values(ascending=False)       # wsparcie każdego produktu
wsp.head(15).plot(kind="barh", figsize=(7, 5)).invert_yaxis()
plt.xlabel("wsparcie (część paragonów z produktem)"); plt.show()
wsp.head(5)

**Pytanie:** dwa produkty są na ponad jednej trzeciej paragonów. Co to oznacza dla reguły „cokolwiek → chleb”: czy wysoka pewność takiej reguły coś mówi?

*Odpowiedź:* …

## Z2. Trzy miary na wybranych parach

Najpierw jedna reguła „ręcznie” (licząc paragony), potem tabela dla kilku par.

In [ ]:
nA = k.liczba(K, "makaron")                         # paragony z makaronem
nAB = k.liczba(K, ["makaron", "sos pomidorowy"])    # paragony z makaronem i sosem
nB = k.liczba(K, "sos pomidorowy")
n = len(K)
pewnosc_recznie = ___ / ___                         # jaka część paragonów z makaronem ma też sos?
lift_recznie = pewnosc_recznie / (___ / n)          # w porównaniu z odsetkiem wszystkich paragonów z sosem
print(nA, nAB, nB, round(pewnosc_recznie, 3), round(lift_recznie, 2))

In [ ]:
pary = [("makaron", "sos pomidorowy"), ("sos pomidorowy", "makaron"), ("kawa", "mleko"),
        ("pieluchy", "chusteczki nawilżane"), ("masło", "chleb"), ("mleko", "chleb")]
pd.DataFrame([para(A, B) for A, B in pary])

**Pytania:** (1) Dlaczego pewność makaron → sos jest inna niż sos → makaron, a lift ten sam? (2) Kawa → mleko ma pewność 0,46, a lift tylko 1,28. Mleko → chleb: pewność 0,40, lift 1,07. Która z tych par to prawdziwy związek, a która głównie popularność następnika?

*Odpowiedź:* …

## Z3. Algorytm Apriori: wszystkie reguły naraz

Funkcja `k.reguly` szuka zbiorów produktów, które występują razem na co najmniej `min_wsparcie` paragonów, i tworzy z nich reguły. Ustaw próg 1% paragonów.

In [ ]:
r = k.reguly(K, min_wsparcie=___, min_lift=1.0)    # 1% paragonów = 0.01
print("reguł:", len(r), " z lift >= 1,2:", (r.lift >= 1.2).sum(), " z lift >= 2:", (r.lift >= 2).sum())
r.head(10)

In [ ]:
# kontrola: Apriori (biblioteka mlxtend) i nasze funkcje dają to samo
w = r[(r.poprzednik == "makaron") & (r.nastepnik == "sos pomidorowy")].iloc[0]
print(round(w.pewnosc, 4), round(k.pewnosc(K, "makaron", "sos pomidorowy"), 4), round(w.lift, 4), round(k.lift(K, "makaron", "sos pomidorowy"), 4))

In [ ]:
# reguły z wysoką pewnością, ale liftem bliskim 1: „popularny następnik”
pop = r[(r.pewnosc >= 0.3) & (r.lift < 1.1)]
print("takich reguł:", len(pop)); pop.head(6)

**Pytanie:** reguł z liftem ≥ 1 jest kilkaset. Które trzy wybrałbyś do rozmowy z kierownikiem sklepu i dlaczego (wsparcie, lift, czy to nie dwa zapisy tej samej zależności)?

*Odpowiedź:* …

## Z4. Cztery pułapki

### a) Próg wsparcia: za niski daje tysiące reguł i przypadkowe rekordy

In [ ]:
for ms in [0.0003, 0.001, 0.003, 0.01, 0.02, 0.05]:
    rr = k.reguly(K, min_wsparcie=ms, min_lift=1.0)
    print(f"min wsparcie {ms:>7}: reguł {len(rr):>6}   najwyższy lift: {rr.poprzednik[0]} → {rr.nastepnik[0]}, lift {rr.lift[0]:.1f}, paragonów {rr.paragony[0]}")

### b) Ile „reguł” powstaje z czystego przypadku?
Przemieszamy losowo każdą kolumnę: popularność produktów zostaje, związki między nimi znikają.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
Kp = K.apply(lambda c: pd.Series(rng.permutation(c.values), index=K.index))
rp = k.reguly(Kp, min_wsparcie=0.01, min_lift=1.0)
print("reguł po przemieszaniu:", len(rp), " z lift >= 1,2:", (rp.lift >= 1.2).sum(), " najwyższy lift:", round(rp.lift.max(), 2))

### c) Pieluchy i piwo: zależy od godziny
Uzupełnij maskę: paragony z godzin 17–19 (kolumna `info.godzina`).

In [ ]:
okno = info.godzina.between(___, ___)
print("wszystkie:", para("pieluchy", "piwo"))
print("17–19:    ", para("pieluchy", "piwo", K[okno]))
print("inne:     ", para("pieluchy", "piwo", K[~okno]))

### d) Grill: wrzesień i październik

In [ ]:
for m, nazwa in [(9, "wrzesień"), (10, "październik")]:
    Km = K[info.data.dt.month == m]
    print(nazwa, para("węgiel drzewny", "kiełbasa", Km))

**Pytania:** (1) Co pokazał test z przemieszaniem? (2) Czy reguła pieluchy → piwo „istnieje”? (3) Czy reguła węgiel → kiełbasa ze stycznia przyszłego roku będzie aktualna?

*Odpowiedź:* …

## Z5. Od reguły do decyzji: ekspozycja czy rabat?

Pomysł 1: postawić sos pomidorowy obok makaronu (koszt ≈ 0 zł). Pomysł 2: rabat 20% na sos dla kupujących makaron.
Założenia (do dyskusji!): marża na sosie 30% ceny 6,99 zł, czyli 2,10 zł; rabat 1,40 zł; 5% klientów z makaronem, ale bez sosu, dokupi sos.

In [ ]:
marza, rabat, konwersja = 2.10, 1.40, 0.05
p = k.potencjal(K, "makaron", "sos pomidorowy", konwersja, marza)
juz_razem = k.liczba(K, ["makaron", "sos pomidorowy"])        # i tak kupują oba
print(p, " juz_razem:", juz_razem)

zysk_ekspozycja = p["dodatkowa_marza_zl"]
koszt_rabatu_dla_juz_kupujacych = juz_razem * rabat
zysk_z_nowych_przy_rabacie = p["dodatkowe_sztuki"] * (marza - rabat)
wynik_rabatu = ___ - ___
print("ekspozycja:", round(zysk_ekspozycja, 2), "zł   rabat:", round(wynik_rabatu, 2), "zł   (dwa miesiące)")

In [ ]:
# przy jakiej konwersji rabat wychodzi na zero?
konwersja_progu = koszt_rabatu_dla_juz_kupujacych / (p["paragony_z_A_bez_B"] * (marza - rabat))
print("rabat się zwraca dopiero przy konwersji:", round(konwersja_progu * 100, 1), "%")

**Pytania:** (1) Dlaczego silna reguła (lift ≈ 6) to argument przeciw rabatowi? (2) Reguła nie mówi, czy ekspozycja zwiększy sprzedaż sosu. Jak to sprawdzić, zanim zmienimy wszystkie sklepy?

*Odpowiedź:* …

## Z6. W domu: dashboard na Render i rekomendacja

1. W terminalu `streamlit run app.py`: sprawdź działanie (Open in Browser).
2. Commit i push (panel Source Control), zielone Actions.
3. Render: **New → Blueprint** → to repozytorium → **Deploy Blueprint** (jak na lab. 5).
4. W `README.md` dopisz **rekomendację dla kierownika sklepu** (5–8 zdań): dwie reguły do działania (z liczbami: paragony, pewność, lift), jedno działanie dla każdej, jedna reguła, której **nie** polecasz, i dlaczego, plan testu A/B. Podaj adres aplikacji.